In [1]:
# 이전 예제 sub classing 네트워크 설계방식 적용
# mnist dataset으로 CNN모델 작성
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [3]:
(x_train, y_train),(x_test,y_test)=tf.keras.datasets.mnist.load_data()

# 구조변경(차원)
print(x_train.shape)  # (60000, 28, 28)
x_train=x_train.reshape((-1,28,28,1)).astype('float32')/255.0    # 타입변경/ 정규화 진행
x_test=x_test.reshape((-1,28,28,1)).astype('float32')/255.0      # 타입변경/ 정규화 진행
print(x_train.shape)  # (60000, 28, 28, 1)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(60000, 28, 28)
(60000, 28, 28, 1)


In [7]:
# model정의
# 사용자 정의 class(모델,레이어,함수:손실,활성화)를 모델 저장시 자동으로 직렬화 시스템에 등록해주는 역할
# @tf.keras.utils.register_keras_serializabler(package='custom')  # 'losses','activation'
class MyMnistCnn(tf.keras.Model):
    def __init__(self,**kwargs):
        super().__init__(**kwargs)
        # Conv block1
        self.conv1=tf.keras.layers.Conv2D(16,(3,3),padding='same',activation='relu')
        self.pool1=tf.keras.layers.MaxPool2D((2,2))
        # Conv block2
        self.conv2=tf.keras.layers.Conv2D(16,(3,3),padding='same',activation='relu')
        self.pool2=tf.keras.layers.MaxPool2D((2,2))
        # Conv block3
        self.conv3=tf.keras.layers.Conv2D(16,(3,3),padding='same',activation='relu')
        self.pool3=tf.keras.layers.MaxPool2D((2,2))

        self.flat=tf.keras.layers.Flatten()

        self.d1=tf.keras.layers.Dense(64,activation='relu')
        self.do1=tf.keras.layers.Dropout(0.3)
        self.d2=tf.keras.layers.Dense(32,activation='relu')
        self.do2=tf.keras.layers.Dropout(0.2)
        self.out=tf.keras.layers.Dense(10,activation='softmax')

    def call(self,inputs,training=False):
        x=self.conv1(inputs)
        x=self.pool1(x)
        x=self.conv2(x)
        x=self.pool2(x)
        x=self.conv3(x)
        x=self.pool3(x)
        x=self.flat(x)
        x=self.d1(x)
        x=self.do1(x,training=True)
        x=self.d2(x)
        x=self.do2(x,training=True)
        return self.out(x)

model=MyMnistCnn()
model.build(input_shape=(None,28,28,1))
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'my_mnist_cnn_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "my_mnist_cnn_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

es=tf.keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True)

history=model.fit(
    x_train,y_train,epochs=100,batch_size=128,validation_split=0.1,
    callbacks=[es],verbose=2
)

# 모델평가
train_loss,train_acc=model.evaluate(x_train,y_train,verbose=0)
test_loss,test_acc=model.evaluate(x_test,y_test,verbose=0)
print(f'train_loss:{train_loss:.4f},train_acc:{train_acc:.4f}')
print(f'test_loss:{test_loss:.4f},test_acc:{test_acc:.4f}')

Epoch 1/100
422/422 - 27s - 63ms/step - accuracy: 0.7804 - loss: 0.6759 - val_accuracy: 0.9385 - val_loss: 0.2140
Epoch 2/100
422/422 - 41s - 96ms/step - accuracy: 0.9438 - loss: 0.2000 - val_accuracy: 0.9595 - val_loss: 0.1445
Epoch 3/100
422/422 - 40s - 96ms/step - accuracy: 0.9603 - loss: 0.1439 - val_accuracy: 0.9688 - val_loss: 0.1160
Epoch 4/100
422/422 - 24s - 56ms/step - accuracy: 0.9674 - loss: 0.1170 - val_accuracy: 0.9750 - val_loss: 0.0870
Epoch 5/100
422/422 - 25s - 58ms/step - accuracy: 0.9715 - loss: 0.1021 - val_accuracy: 0.9787 - val_loss: 0.0815
Epoch 6/100
422/422 - 40s - 95ms/step - accuracy: 0.9746 - loss: 0.0918 - val_accuracy: 0.9818 - val_loss: 0.0784
Epoch 7/100
422/422 - 41s - 96ms/step - accuracy: 0.9771 - loss: 0.0823 - val_accuracy: 0.9785 - val_loss: 0.0800
Epoch 8/100
422/422 - 41s - 98ms/step - accuracy: 0.9792 - loss: 0.0747 - val_accuracy: 0.9830 - val_loss: 0.0701
Epoch 9/100
422/422 - 41s - 97ms/step - accuracy: 0.9811 - loss: 0.0687 - val_accuracy: 